In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

In [ ]:
!nvidia-smi

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


sys.path.insert(1, '../src/')

from cellpose import models, core, io, plot
from Basic import *
from image_lib import *

In [ ]:
is_laptop = True

root0 = "../../colaboracoes/deOcesano/samples/"

os.listdir(root0)

In [ ]:
plates = [x for x in os.listdir(root0) if x.startswith('Plate')]
plates

In [ ]:
i=0
plate = plates[i]
plate

In [ ]:
root_plate = os.path.join(root0, plate)

root_exps = [x for x in os.listdir(root_plate) if os.path.isdir( os.path.join(root_plate, x) )]
root_exps

print(root_exps)

root_plate = os.path.join(root_plate, root_exps[1])
print(root_plate)

## Choose an image to classifier

In [ ]:
image_chose = os.listdir(root_plate)[1]
image_path = os.path.join(root_plate, image_chose)

print(image_chose)
img = io.imread(image_path)
plt.imshow(img)
img.shape

## Cellpose

In [ ]:
gpu_disponible = torch.cuda.is_available()

if gpu_disponible: print('GPU disponível')
else: print('GPU não disponível')

In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '2' # @param ['None', 0, 1, 2, 3, 4, 5]

selected_channels = []

for c in [first_channel, second_channel, third_channel]:
    if c == 'None': continue
    if int(c) > img.shape[-1]: assert False, 'invalid channel index, must have index greater or equal to the number of channels'
    if c != 'None': selected_channels.append(int(c))

selected_channels

### Processing Cellpose

In [ ]:
%%time

model = models.CellposeModel(gpu=True)

img_selected_channels = np.zeros_like(img)
img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]

flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(
    img_selected_channels,
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize}
)

In [ ]:
fig = plt.figure(figsize=(12,12))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Configuração ---
OUTPUT_DIR = "../../colaboracoes/deOcesano/segment/Plate1848/FACT - 1%SF"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📂 Salvando segmentos em: {OUTPUT_DIR}")

# --- 2. Processamento ---
# Obtém todos os IDs únicos das células (exclui o 0 que é o fundo)
cell_ids = np.unique(masks)
cell_ids = cell_ids[cell_ids != 0]

print(f"Iniciando o recorte de {len(cell_ids)} células...")

count = 0
for cell_id in cell_ids:
    # A. Encontra as coordenadas (pixels) que pertencem a esta célula
    # Retorna uma tupla de arrays (indices_y, indices_x)
    coords = np.where(masks == cell_id)
    
    # B. Define a "Caixa Delimitadora" (Bounding Box)
    y_min, y_max = np.min(coords[0]), np.max(coords[0])
    x_min, x_max = np.min(coords[1]), np.max(coords[1])
    
    # C. Recorta a imagem original na região da célula
    # Nota: Adicionamos +1 no max porque o slice do Python é exclusivo no final
    cell_crop = img_selected_channels[y_min:y_max+1, x_min:x_max+1].copy()
    
    # D. (Opcional) Aplicar máscara para deixar o fundo do recorte preto
    # Isso garante que partes de células vizinhas não apareçam no quadrado
    cell_mask_local = (masks[y_min:y_max+1, x_min:x_max+1] == cell_id)
    
    # Se a imagem for colorida/multicanal, ajustamos a máscara para multiplicar corretamente
    if cell_crop.ndim == 3:
        cell_mask_local = cell_mask_local[..., np.newaxis]
        
    # Zera tudo que não for a célula atual dentro do quadrado recortado
    cell_crop = cell_crop * cell_mask_local

    # --- 3. Salvamento ---
    filename = f"cell_{cell_id:04d}.tif" # Ex: cell_0001.tif
    save_path = os.path.join(OUTPUT_DIR, filename)
    
    # Usa o OpenCV para salvar. Se der erro de cor invertida (BGR vs RGB), 
    # use cv2.cvtColor(cell_crop, cv2.COLOR_RGB2BGR) antes de salvar.
    # Para imagens científicas monocromáticas ou TIFF, geralmente é direto.
    cv2.imwrite(save_path, cell_crop)
    count += 1

print(f"✅ Concluído! {count} segmentos individuais foram salvos.")